# SDF-GAN Autoresearch: Experiment Analysis

Analysis of autonomous experiment results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('results.tsv', sep='\t')
df['valid_sharpe'] = pd.to_numeric(df['valid_sharpe'], errors='coerce')
df['test_sharpe'] = pd.to_numeric(df['test_sharpe'], errors='coerce')
df['valid_ev'] = pd.to_numeric(df['valid_ev'], errors='coerce')
df['status'] = df['status'].str.strip().str.upper()

print(f'Total experiments: {len(df)}')
df.head(10)

In [ ]:
counts = df['status'].value_counts()
print('Experiment outcomes:')
print(counts.to_string())

n_keep = counts.get('KEEP', 0)
n_discard = counts.get('DISCARD', 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f'\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}')

In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
print(f'KEPT experiments ({len(kept)} total):\n')
for i, row in kept.iterrows():
    sr = row['valid_sharpe']
    desc = row['description']
    print(f'  #{i:3d}  valid_sharpe={sr:.6f}  test_sharpe={row["test_sharpe"]:.6f}  {desc}')

## Valid Sharpe Over Time

Track how the best (kept) valid_sharpe evolves. The running maximum shows the frontier.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df['status'] != 'CRASH'].copy().reset_index(drop=True)
baseline_sharpe = valid.loc[0, 'valid_sharpe']

# Plot discarded as faint dots
disc = valid[valid['status'] == 'DISCARD']
ax.scatter(disc.index, disc['valid_sharpe'],
           c='#cccccc', s=12, alpha=0.5, zorder=2, label='Discarded')

# Plot kept as prominent green dots
kept_v = valid[valid['status'] == 'KEEP']
ax.scatter(kept_v.index, kept_v['valid_sharpe'],
           c='#2ecc71', s=50, zorder=4, label='Kept', edgecolors='black', linewidths=0.5)

# Running maximum
kept_mask = valid['status'] == 'KEEP'
kept_idx = valid.index[kept_mask]
kept_sharpe = valid.loc[kept_mask, 'valid_sharpe']
running_max = kept_sharpe.cummax()
ax.step(kept_idx, running_max, where='post', color='#27ae60',
        linewidth=2, alpha=0.7, zorder=3, label='Running best')

# Label each kept experiment
for idx, sr in zip(kept_idx, kept_sharpe):
    desc = str(valid.loc[idx, 'description']).strip()
    if len(desc) > 45:
        desc = desc[:42] + '...'
    ax.annotate(desc, (idx, sr),
                textcoords='offset points',
                xytext=(6, 6), fontsize=8.0,
                color='#1a7a3a', alpha=0.9,
                rotation=30, ha='left', va='bottom')

best = kept_sharpe.max()
ax.set_xlabel('Experiment #', fontsize=12)
ax.set_ylabel('Validation Sharpe (higher is better)', fontsize=12)
ax.set_title(f'SDF-GAN Autoresearch: {len(df)} Experiments, {len(kept_v)} Kept', fontsize=14)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.2)

margin = (best - baseline_sharpe) * 0.15 if best > baseline_sharpe else 0.05
ax.set_ylim(baseline_sharpe - margin, best + margin)

plt.tight_layout()
plt.savefig('progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to progress.png')

## Summary Statistics

In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
baseline_sharpe = df.iloc[0]['valid_sharpe']
best_sharpe = kept['valid_sharpe'].max()
best_row = kept.loc[kept['valid_sharpe'].idxmax()]

print(f'Baseline valid_sharpe:  {baseline_sharpe:.6f}')
print(f'Best valid_sharpe:      {best_sharpe:.6f}')
print(f'Total improvement:      {best_sharpe - baseline_sharpe:+.6f} ({(best_sharpe - baseline_sharpe) / abs(baseline_sharpe) * 100:+.2f}%)')
print(f'Best experiment:        {best_row["description"]}')
print()
print('Cumulative improvements:')
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row['description']).strip()
    print(f'  Experiment #{row["index"]:3d}: sharpe={row["valid_sharpe"]:.6f}  {desc}')

## Top Hits (Kept Experiments by Improvement)

In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
kept['prev_sharpe'] = kept['valid_sharpe'].shift(1)
kept['delta'] = kept['valid_sharpe'] - kept['prev_sharpe']

hits = kept.iloc[1:].copy()
hits = hits.sort_values('delta', ascending=False)

print(f'{"Rank":>4}  {"Delta":>8}  {"Sharpe":>10}  Description')
print('-' * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f'{rank:4d}  {row["delta"]:+.6f}  {row["valid_sharpe"]:.6f}  {row["description"]}')

print(f'\n{"":>4}  {hits["delta"].sum():+.6f}  {"":>10}  TOTAL improvement over baseline')